# 03 — Radio / RM Synthesis
## Generate synthetic cubes, run RM synthesis, inspect Q/U structure

This notebook:
1. Generates synthetic Stokes Q/U cubes for both DM models
2. Runs RM synthesis to produce Faraday depth spectra
3. Inspects the spatial RM maps visually
4. Plots the FDF (Faraday Dispersion Function) at selected positions

**This notebook does not make any diagnostic measurement.**
It verifies the data pipeline works end-to-end before the transect is cut.

---

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from astropy.io import fits

sys.path.insert(0, '../modules')
from constants import RA0, DEC0, DM_NW, DM_SE, GAS_NW, GAS_SE
from synthetic import generate_cubes, OUT_DIR, SIZE_PIX

SYNTH_DIR = Path(OUT_DIR)
BASE_DIR  = Path('/media/rendier/0123-4567/bullet_cluster')

# Check for real MGCLS cubes first
real_q = BASE_DIR / 'radio/meerkat/MGCLS_1E0657-558_Q_cube.fits'
USE_REAL = real_q.exists()
print(f'Real MGCLS cubes available: {USE_REAL}')
if USE_REAL:
    print(f'  Will use: {real_q}')
else:
    print('  Will use synthetic cubes')
    print('  Register at: https://archive.sarao.ac.za/')
    print('  Proposal: SSV-20180624-FC-01')

In [ ]:
# Generate synthetic cubes for both DM models
# Skip if already present
for model in ['wave', 'particle']:
    print(f'\n--- DM model: {model} ---')
    result = generate_cubes(dm_model=model, overwrite=False)
    print(f'  Q cube: {result[1]}')

In [ ]:
# Load Q cube and inspect
model = 'wave'
with fits.open(SYNTH_DIR / f'synthetic_Q_{model}.fits') as h:
    Q = h[0].data
with fits.open(SYNTH_DIR / f'synthetic_U_{model}.fits') as h:
    U = h[0].data
freqs = np.loadtxt(SYNTH_DIR / f'synthetic_freqs_{model}.dat')

print(f'Q cube shape (Nfreq, Ny, Nx): {Q.shape}')
print(f'Frequency range: {freqs[0]/1e6:.0f} – {freqs[-1]/1e6:.0f} MHz')
print(f'Q range: [{Q.min():.4f}, {Q.max():.4f}] Jy/beam')
print(f'U range: [{U.min():.4f}, {U.max():.4f}] Jy/beam')

In [ ]:
# Faraday Dispersion Function at three positions: DM_NW, GAS_NW, relic
from astropy.wcs import WCS

with fits.open(SYNTH_DIR / f'synthetic_Q_{model}.fits') as h:
    wcs = WCS(h[0].header, naxis=2)

lambda2 = (2.998e8 / freqs) ** 2   # m²

phi_arr = np.arange(-300, 301, 2.0)  # rad/m² search range

positions = [
    ('DM NW',  DM_NW['ra'],  DM_NW['dec']),
    ('Gas NW', GAS_NW['ra'], GAS_NW['dec']),
    ('DM SE',  DM_SE['ra'],  DM_SE['dec']),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (label, ra, dec) in zip(axes, positions):
    xi, yi = [int(round(v)) for v in wcs.all_world2pix([[ra, dec]], 0)[0]]
    xi = np.clip(xi, 0, Q.shape[2]-1)
    yi = np.clip(yi, 0, Q.shape[1]-1)
    P = Q[:, yi, xi] + 1j * U[:, yi, xi]
    FDF = np.array([np.nansum(P * np.exp(-2j * phi * lambda2))
                     for phi in phi_arr])
    FDF /= np.sum(np.isfinite(Q[:, yi, xi]))
    peak_phi = phi_arr[np.argmax(np.abs(FDF))]
    ax.plot(phi_arr, np.abs(FDF), 'k', lw=0.8)
    ax.axvline(peak_phi, color='red', ls='--', lw=1, label=f'Peak: {peak_phi:.0f} rad/m²')
    ax.set_xlabel('Faraday depth (rad/m²)')
    ax.set_ylabel('|FDF|')
    ax.set_title(f'{label}\n(pixel {xi},{yi})')
    ax.legend(fontsize=8)

plt.suptitle(f'Faraday Dispersion Functions — synthetic {model} DM model', y=1.02)
plt.tight_layout()
plt.savefig('../output/fdf_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/fdf_comparison.png')

In [ ]:
# RM peak map for both models — shows where the synthetic Faraday screen is
from scipy.ndimage import gaussian_filter

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, dm_model in zip(axes, ['wave', 'particle']):
    with fits.open(SYNTH_DIR / f'synthetic_Q_{dm_model}.fits') as h:
        Qm = h[0].data
    with fits.open(SYNTH_DIR / f'synthetic_U_{dm_model}.fits') as h:
        Um = h[0].data

    lambda2_m = (2.998e8 / freqs) ** 2
    # Quick RM peak map: just sum over φ with matched filter (coarse but fast)
    phi_test = np.arange(-200, 201, 5.0)
    P = Qm + 1j * Um   # (Nfreq, Ny, Nx)
    # For each spatial pixel find argmax FDF — sample every 4th pixel for speed
    step = 4
    ny, nx = Qm.shape[1], Qm.shape[2]
    rm_map = np.zeros((ny//step, nx//step))
    for iy_s in range(ny//step):
        iy = iy_s * step
        for ix_s in range(nx//step):
            ix = ix_s * step
            spec = P[:, iy, ix]
            fdf = np.abs([np.nansum(spec * np.exp(-2j * phi * lambda2_m))
                          for phi in phi_test])
            rm_map[iy_s, ix_s] = phi_test[np.argmax(fdf)]

    im = ax.imshow(rm_map, origin='lower', cmap='RdBu_r', vmin=-150, vmax=150)
    plt.colorbar(im, ax=ax, label='RM peak (rad/m²)')
    ax.set_title(f'RM peak map — {dm_model} DM\n(downsampled ×{step})')
    # Mark positions
    for label, ra, dec, col in [
        ('DM NW', DM_NW['ra'],  DM_NW['dec'],  'blue'),
        ('DM SE', DM_SE['ra'],  DM_SE['dec'],  'blue'),
        ('Gas NW', GAS_NW['ra'], GAS_NW['dec'], 'red'),
    ]:
        xi, yi = [v / step for v in wcs.all_world2pix([[ra, dec]], 0)[0]]
        ax.plot(xi, yi, 'x', color=col, ms=8, mew=2)
        ax.annotate(label, (xi, yi), fontsize=6, color=col,
                    xytext=(3, 3), textcoords='offset points')

plt.suptitle('Synthetic RM maps — wave vs particle DM')
plt.tight_layout()
plt.savefig('../output/rm_maps_synthetic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/rm_maps_synthetic.png')
print()
print('The particle map should show excess RM at DM NW and DM SE.')
print('The wave map should show smooth RM tracking the gas, not the DM.')